# Stochastic Calculus, Step by Step · Lesson 4: The normal distribution

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. The standard normal · 3. Standardising, and symmetry · 4. Moments and the moment generating function · 5. The exponential of a normal · 6. Sums of independent normals · 7. Completing the square · 8. E|Z| · 9. Exercises · 10. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 4,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. The standard normal

$\varphi(z) = \frac{1}{\sqrt{2\pi}}e^{-z^2/2}$. Check numerically that its area is 1, its mean 0 and its variance 1, and tabulate the bell's height and the distribution function $N$ at a few points.

In [ ]:
grid = np.linspace(-12, 12, 240_001)
dz = grid[1] - grid[0]
phi_grid = np.exp(-grid**2 / 2) / math.sqrt(2 * math.pi)
out["stdnorm"] = {
    "area": float(phi_grid.sum() * dz), "mean": float((grid * phi_grid).sum() * dz),
    "var": float((grid**2 * phi_grid).sum() * dz), "sqrt2pi": math.sqrt(2 * math.pi),
    "heights": [{"z": z, "h": math.exp(-z**2 / 2)} for z in (0, 1, 2, 3, 4)],
    "Ntable": [{"x": x, "N": N(x)} for x in (0.0, 0.5, 1.0, 1.645, 1.96, 2.0, 3.0)],
}
out["stdnorm"]

## 3. Standardising, and symmetry

If $X \sim N(\mu, s^2)$ then $Z = (X - \mu)/s \sim N(0, 1)$, and $N(-x) = 1 - N(x)$. Take the share's one-year log return under the risk-neutral rule of Lesson 5, $L = \ln(S_T/S_0) \sim N\big((r - \tfrac12\sigma^2)T, \sigma^2T\big)$. The chance the share ends above the strike:

In [ ]:
m_L, s_L = (r - 0.5 * sigma**2) * T, sigma * math.sqrt(T)
lnK = math.log(K / S0)
z_K = (lnK - m_L) / s_L
rng = np.random.default_rng(seed)
Z = rng.standard_normal(2_000_000)
L = m_L + s_L * Z
out["std"] = {"m_L": m_L, "s_L": s_L, "lnK": lnK, "z_K": z_K, "N_zK": N(z_K), "p_above": 1 - N(z_K),
              "p_above_sym": N(-z_K), "p_above_mc": float((L > lnK).mean())}
out["std"]

## 4. Moments and the moment generating function

$M(u) = E[e^{uZ}] = e^{u^2/2}$ (completing the square). Its derivatives at 0 are the moments: $E[Z^2] = 1$, $E[Z^4] = 3$, $E[Z^6] = 15$.

In [ ]:
mom = {"m1": float(Z.mean()), "m2": float((Z**2).mean()), "m3": float((Z**3).mean()), "m4": float((Z**4).mean()),
       "m6": float((Z**6).mean()), "m4_se": float((Z**4).std() / math.sqrt(Z.size))}
mom["rows"] = [{"k": k, "exact": e, "mc": mom[f"m{k}"]} for k, e in ((1, 0), (2, 1), (3, 0), (4, 3), (6, 15))]
out["moments"] = mom
mom

## 5. The exponential of a normal

$E[e^{\sigma Z}] = e^{\sigma^2/2}$, not $e^{E[\sigma Z]} = 1$. A two-point variable with the same mean and variance, $\pm\sigma$ with probability one half each, already shows why: the up move gains more than the down move loses. And with $S_T = S_0e^L$, $E[S_T] = S_0e^{m_L + s_L^2/2} = S_0e^{rT}$.

In [ ]:
ST = S0 * np.exp(L)
out["expo"] = {"mgf_sigma": math.exp(sigma**2 / 2), "mgf_sigma_mc": float(np.exp(sigma * Z).mean()),
               "up": math.exp(sigma) - 1, "down": 1 - math.exp(-sigma),
               "two_point": (math.exp(sigma) + math.exp(-sigma)) / 2,
               "ES_exact": S0 * math.exp(r * T), "ES_mc": float(ST.mean()),
               "ES_naive": S0 * math.exp(m_L)}
out["expo"]

## 6. Sums of independent normals

Independent normals add to a normal: means add, variances add. A year of 252 independent daily log returns, each $N\big((r - \tfrac12\sigma^2)\Delta t, \sigma^2\Delta t\big)$, adds up to the yearly $N\big(r - \tfrac12\sigma^2, \sigma^2\big)$.

In [ ]:
days = 252
daily = rng.normal(m_L / days, s_L / math.sqrt(days), (100_000, days))
year = daily.sum(axis=1)
out["sum"] = {"days": days, "mean": float(year.mean()), "sd": float(year.std()), "mean_th": m_L, "sd_th": s_L,
              "daily_mean": m_L / days, "daily_var": s_L**2 / days, "daily_sd": s_L / math.sqrt(days)}
out["sum"]

## 7. Completing the square

$$e^{uz}\varphi(z) = e^{u^2/2}\varphi(z - u):$$

tilting the normal density by $e^{uz}$ shifts it right by $u$ and scales it by $e^{u^2/2}$. Integrating over $z > a$:

$$E\big[e^{uZ}\mathbf 1\{Z > a\}\big] = e^{u^2/2}\,N(u - a).$$

With $S_T = S_0e^{(r - \frac12\sigma^2)T + \sigma\sqrt T Z}$, this gives the first term of the Black-Scholes formula (Lesson 20): $e^{-rT}E[S_T\mathbf 1\{S_T > K\}] = S_0N(d_+)$ with $d_+ = \big(\ln(S_0/K) + (r + \tfrac12\sigma^2)T\big)/(\sigma\sqrt T)$.

In [ ]:
d_minus = (math.log(S0 / K) + (r - 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
d_plus = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
term = math.exp(-r * T) * float((ST * (ST > K)).mean())
z = np.linspace(-4, 5, 361)
phi = np.exp(-z**2 / 2) / math.sqrt(2 * math.pi)
u_fig = 1.0
out["square"] = {"d_minus": d_minus, "d_plus": d_plus, "sigma_sqrtT": sigma * math.sqrt(T), "N_d_plus": N(d_plus),
                 "S0N": S0 * N(d_plus), "mc": term,
                 "fig": {"z": z.round(3).tolist(), "phi": phi.round(6).tolist(),
                         "tilted": (np.exp(u_fig * z) * phi).round(6).tolist(),
                         "shifted": (math.exp(u_fig**2 / 2) * np.exp(-(z - u_fig)**2 / 2) / math.sqrt(2 * math.pi)).round(6).tolist(),
                         "u": u_fig, "scale": math.exp(u_fig**2 / 2),
                         "ticks": [{"x": v, "label": f"{v:g}".replace("-", "−")} for v in range(-4, 5)]}}
assert abs(term - S0 * N(d_plus)) < 0.1
{k: v for k, v in out["square"].items() if k != "fig"}

## 8. E|Z|

$E|Z| = 2\int_0^\infty z\varphi(z)\,dz = \frac{2}{\sqrt{2\pi}}\int_0^\infty ze^{-z^2/2}\,dz = \sqrt{2/\pi}$, using Lesson 1's substitution. Lesson 14 uses it to show that a Brownian path has infinite length.

In [ ]:
out["absz"] = {"exact": math.sqrt(2 / math.pi), "mc": float(np.abs(Z).mean())}
out["absz"]

## 9. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 1: Solutions*.

**Exercise 1.** Let $X \sim N(100, 20^2)$. Compute $P(X > 105)$ and $P(90 < X < 110)$ using $N$ and its symmetry.

**Exercise 2.** Use the moment generating function $M(u) = e^{u^2/2}$ to find $E[Z^6]$. (Expand $M$ as a power series and match coefficients.)

**Exercise 3.** Use moment generating functions to show that if $X_1, \dots, X_n$ are independent $N(\mu, s^2)$, then $X_1 + \dots + X_n \sim N(n\mu, ns^2)$.

**Exercise 4.** Show by completing the square that $E[e^{uZ}\mathbf 1\{Z > a\}] = e^{u^2/2}N(u - a)$. Evaluate it for $u = 0.2$, $a = 0$.

**Exercise 5.** Let $X = +1$ or $-1$ with probability one half each. Find its moment generating function, and use it to find $E[X^2]$ and $E[X^4]$. Compare $E[X^4]$ with $E[Z^4] = 3$: what does the difference say about the tails?

**Exercise 6.** Find the moment generating function of $X \sim N(\mu, s^2)$ directly from its density, by completing the square, without writing $X = \mu + sZ$.

**Exercise 7.** (Code) Estimate $E[Z^4]$ and $E|Z|$ by simulation with $10^3$, $10^5$ and $10^7$ samples. Which settles faster, and why?

**Exercise 8.** (Code) Check $e^{-rT}E[S_T\mathbf 1\{S_T > K\}] = S_0N(d_+)$ by simulation with the course's numbers.

## 10. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")